# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. ([1,2,3], [0, 1, 1]), деление делит каждый элемент списка на 2, и на 4 без остатка
2. [4 4 4], сумма по строкам, у нас 3 строки [1 1 1 1], сумма каждой 4
3. [0,2,4] , [0,1,2] + [0,1,2] = [0,2,4]
4. 6, суммируем корень каждого элемента, это 1+2+3=6
5. 7.5, 0^2 + 1^2 + 2^2 + 3^2 + 4^2 = 1+4+9+16 = 30, среднее 30/4 = 7.5

In [4]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3).reshape(3, 1))
print(np.arange(3))
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0]
 [1]
 [2]]
[0 1 2]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [5]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [6]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [7]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [13]:
# 1
print("в чистых данных:", clean.sum())
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])
table = np.array([amount[clean & (month == m)].sum() for m in months])
print("Выручка за сентябрь:", table[0], "Выручка за сентябрь:", table[1], "Изменение в %:" , (table[1]/table[0] - 1)*100 )
# Не совпадает потому что у нас очищенная таблица, без повторений и тестов

в чистых данных: 4739
Выручка за сентябрь: 35243410.0 Выручка за сентябрь: 34419520.0 Изменение в %: -2.337713632137184


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [37]:
# 2
categories = np.unique(cat)
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                  for c in categories])
categories_sum = table.sum(axis=1)
categories_fall = (table[:, 1] / table[:, 0] - 1)*100

print("Категория которая приносит больше всего выручки за 2 месяца это", categories[categories_sum.argmax()] , "на сумму:" , categories_sum.max())
print("Категория которая упала сильнее всех это", categories[categories_fall.argmin()], "на " , categories_fall.min() , "%")

Категория которая приносит больше всего выручки за 2 месяца это Электроника на сумму: 34370430.0
Категория которая упала сильнее всех это Одежда на  -7.214604400519253 %


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [38]:
# 3
table_2 = np.array([[amount[clean & (cat == c) & (month == m)].shape[0] for m in months]
                  for c in categories])
for c, r in zip(categories, table_2):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")

new_table = table.sum(axis=1) / table_2.sum(axis=1)
print("Категория с самым высоким средним чеком " , categories[new_table.argmax()], ":" , new_table.max())

Дом                 476          476
Книги               265          299
Красота             422          435
Одежда              678          652
Электроника          531          505
Категория с самым высоким средним чеком  Электроника : 33176.09073359073


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [49]:
# 4
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
new_table = table / table.sum(axis=0)
check = new_table.sum(axis=0)
print("Проверка суммы столбцов:" , check) # проверяем чтобы вышло 1. 1.
for i, c in enumerate(cities):
  if c == "Шымкент":
    print("Доля", c , "изменилась на ", (new_table[i, 1] - new_table[i, 0])*100, "%")

Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
Проверка суммы столбцов: [1. 1.]
Доля Шымкент изменилась на  -1.1855825342961728 %


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [57]:
# 5
table_change = table[:, 1] - table[:, 0]
new_table = table_change / table_change.sum() * 100
for i, c in enumerate(cities):
  if c == "Шымкент":
    print( c , "Дает процентов падения на", new_table[i], "%.", ", стоит посмотреть спрос и активность продаж в Шымкенте")

Шымкент Дает процентов падения на 62.46464940708104 %. , стоит посмотреть спрос и активность продаж в Шымкенте


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [61]:
# 6
share = table / table.sum(axis=1, keepdims=True)
print(share)
# Код падает потому что нам нужно размерность матриц была правильной, для этого используем keepDims=True

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [69]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
print("Заказов с |z| > 3: " , (z[z>3].shape[0] + z[z<-3].shape[0]), "Доля:" , ((z[z>3].shape[0] + z[z<-3].shape[0]) / z.shape[0] * 100) , "%")
print(z.mean(), z.std())

Заказов с |z| > 3:  87 Доля: 1.8358303439544206 %
7.04695264417065e-17 1.0



## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [78]:
# 8
a = days[clean & ~np.isnan(days)]
b = amount[clean & ~np.isnan(days)]
X = np.column_stack((b,a))
new_table = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(new_table)
print(new_table[:,0].min(), new_table[:,0].max(), new_table[:,1].min(), new_table[:,1].max())

[[0.05313765 0.        ]
 [0.02103817 0.14285714]
 [0.06072874 0.28571429]
 ...
 [0.07742915 0.14285714]
 [0.02111047 0.14285714]
 [0.1025882  0.14285714]]
0.0 1.0 0.0 1.0


Модель будет смотреть корреляцию признаков друг к другу, и если у одного будет рендж (1, 1000000) а у другого (1, 31) то один признак может повлиять сильнее чем второй, поэтому приводим к одному масштабу чтобы можно было различить влияние признаков друг на друга.
